## 1.Explore and cleaning data

In [7]:
import pandas as pd
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu' 
DEVICE

'cuda'

In [8]:
import torch

In [9]:
raw_df = pd.read_csv('../data/raw/SNAPFOOD.csv')

In [10]:
#just checkout to see what we have 
print(raw_df.head())
print("="*50)
print(raw_df.info())
print("="*50)
print(raw_df.describe())
print("="*50)
print(raw_df.isnull().sum())
raw_df.dropna(inplace=True)

   Unnamed: 0                                            comment  label  \
0           0   غذا خیلی سرد بود در صورتیکه فاصله ما خیلی کم است    SAD   
1           1     بهتره بتونیم ران یا سینه رو خودمون انتخاب کنیم  HAPPY   
2           2  غذا بد بود حالم خیییییلی بده. دل دردو دل پیچه....    SAD   
3           3  با سلام سابق بر این بسته بندی از کیفیت بهتری ب...    SAD   
4           4                          سلام، خیلی ممنون و متشکرم  HAPPY   

   label_id  
0       1.0  
1       0.0  
2       1.0  
3       1.0  
4       0.0  
<class 'pandas.DataFrame'>
RangeIndex: 52110 entries, 0 to 52109
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  52110 non-null  int64  
 1   comment     52110 non-null  str    
 2   label       52110 non-null  str    
 3   label_id    52110 non-null  float64
dtypes: float64(1), int64(1), str(2)
memory usage: 9.8 MB
None
         Unnamed: 0      label_id
count  52110.000000  5211

In [11]:
#delete useless columns
raw_df.drop(columns=['Unnamed: 0'],inplace=True)


In [12]:
#check for duplicated record
print(raw_df.duplicated().sum())
#look like we have some np.int64(4909)
raw_df.drop_duplicates(inplace=True)


0


In [15]:
#check lable
raw_df['label_id'].unique()
#its look like 2 sad & happy 
raw_df['label_id'] = raw_df['label_id'].apply(lambda x: int(x))
raw_df['comment'] = raw_df['comment'].str.strip()

In [16]:
raw_df['label_id'].value_counts()
#the label_id column is balanced 

label_id
0    26236
1    25874
Name: count, dtype: int64

## 2.Tokenization and make processed data

In [19]:
import os
os.environ["TIKTOKEN_CACHE_DIR"] = "C:/tiktoken_cache"#iran internet connection ....
import tiktoken
tokenizer = tiktoken.get_encoding("cl100k_base")

In [20]:
processed_df = pd.DataFrame({
    'tokens' : raw_df['comment'].apply(lambda x : tokenizer.encode(x)),
    'label'  : raw_df['label_id']
}).reset_index(drop=True)

In [21]:
processed_df

,tokens,label
0,"[82878, 56434, 5821, 75415, 14728, 8700, 14728...",1
1,"[22071, 16552, 14628, 11318, 16552, 28946, 146...",0
2,"[82878, 56434, 5821, 28946, 13628, 28946, 7052...",1
3,"[22071, 5821, 60942, 8700, 50488, 60942, 71704...",1
4,"[20665, 8700, 50488, 69885, 75415, 14728, 8700...",0
...,...,...
52105,"[14728, 33411, 14728, 13258, 40797, 82868, 146...",0
52106,"[22071, 16552, 24252, 12942, 28590, 24102, 386...",0
52107,"[36344, 14728, 8700, 14728, 40534, 5821, 40797...",0
52108,"[21604, 12942, 28590, 17607, 24102, 66498, 165...",0


In [22]:
max_len = 128
padded_tokens = [tokens[:max_len] + [0]*(max_len - len(tokens))
                for tokens in processed_df['tokens']]  

texts_tensor = torch.tensor(padded_tokens)
labels_tensor = torch.tensor(processed_df['label'].values) 

torch.save(texts_tensor,'../data/processed/texts.pt')
torch.save(labels_tensor,'../data/processed/labels.pt')

## 3.Create Dataloader

In [23]:
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader , TensorDataset

texts_tensor = torch.load('../data/processed/texts.pt')
labels_tensor = torch.load('../data/processed/labels.pt')

#we balance data score here (not very necessary for this dataset)
class_weight = compute_class_weight(
    'balanced',
    classes=torch.unique(labels_tensor).numpy(),
    y=labels_tensor.numpy()
)

weight_tensor = torch.tensor(class_weight, dtype=torch.float)

X_train, X_test, y_train, y_test = train_test_split(texts_tensor, labels_tensor, test_size=0.2, random_state=42,stratify=labels_tensor)

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(
    train_dataset, 
    batch_size=64,  
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    prefetch_factor=2  
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=64,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    prefetch_factor=2
)

## 4.Make LSTM Model

In [33]:
import torch.nn as nn

import torch
import torch.nn as nn

class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim, 
                 n_layers=2, dropout=0.5, pad_idx=0):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, n_layers, 
                           batch_first=True, 
                           dropout=dropout if n_layers > 1 else 0,
                           bidirectional=True)
        # Add layer normalization
        self.layer_norm = nn.LayerNorm(hidden_dim * 2)
        # Add more capacity
        self.fc1 = nn.Linear(hidden_dim * 2, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        output, (hidden, cell) = self.lstm(embedded)
        
        # Better way to combine bidirectional hidden states
        hidden_last = torch.cat((hidden[-2, :, :], hidden[-1, :, :]), dim=1)
        hidden_last = self.layer_norm(hidden_last)
        hidden_last = self.relu(self.fc1(hidden_last))
        hidden_last = self.dropout(hidden_last)
        
        return self.fc2(hidden_last)

vocab_size = tokenizer.n_vocab
lstm_model = SentimentLSTM(
    vocab_size=vocab_size,
    embed_dim=64, 
    hidden_dim=64,  
    output_dim=2,
    n_layers=3,  
    dropout=0.5,
    pad_idx=0
).to(DEVICE)

## Training

In [34]:
def create_optim_loss(model : nn.Module , lr:float , weight_tensor : torch.tensor = None , device : str = None):
    """ return optim , loss_fn """
    optim = torch.optim.AdamW(params=model.parameters(),lr=lr)
    if weight_tensor != None:
        loss_fn = nn.CrossEntropyLoss(weight=weight_tensor.to(device))
    else :
        loss_fn = nn.CrossEntropyLoss() 
    return optim , loss_fn    

In [35]:
import time
def train_step(model : nn.Module , dataloader : torch.utils.data.dataloader  , optimizer : torch.optim , loss_fn : nn.Module ,
               accuracy_fn : None ,device : str = 'cuda' ):
    model.train()
    train_loss , train_acc =0,0
    start_time = time.time()
    for batch,(X,y) in enumerate(dataloader):
        X , y = X.to(device) , y.to(device)

        y_pred = model(X)
        loss = loss_fn(y_pred,y)
        train_acc += accuracy_fn(y,y_pred.argmax(dim=1))
        train_loss += loss.item()
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if batch % 200 == 0:
            print(f"  Batch {batch}, time: {time.time() - start_time:.1f}s")
            
    train_acc/=len(dataloader)
    train_loss/=len(dataloader)
        
    return train_loss,train_acc
    

In [36]:
def test_step(model : nn.Module , dataloader : torch.utils.data.dataloader  , loss_fn : nn.Module ,accuracy_fn : None ,
              device : str = 'cuda' ):
    model.eval()
    with torch.inference_mode():
        
        test_loss,test_acc=0,0
        for batch,(X,y) in enumerate(dataloader):
            X,y=X.to(device),y.to(device)

            y_pred=model(X)
        
            loss=loss_fn(y_pred,y)
        
            test_loss+=loss.item()
            test_acc+=accuracy_fn(y,y_pred.argmax(dim=1))

        test_loss/=len(dataloader)                 
        test_acc/=len(dataloader)
        return test_loss,test_acc
    

In [37]:
from tqdm.auto import tqdm
from timeit import default_timer as timer

def Train(model : nn.Module,train_dataloader:torch.utils.data.dataloader, test_dataloader:torch.utils.data.dataloader, 
          optimizer : torch.optim , loss_fn : nn.Module , accuracy_fn : None , epochs :int = 5 , device : str = 'cuda' ):
    
    start = timer()
    
    results={
        "train_loss":[],
        "train_acc":[],
        "test_loss":[],
        "test_acc":[]}
    
    for epoch in tqdm(range(epochs)):
        
        train_loss,train_acc=train_step(model=model,
                                        dataloader=train_dataloader,
                                        optimizer=optimizer,
                                        loss_fn=loss_fn,
                                        accuracy_fn=accuracy_fn,
                                        device=device)
        test_loss,test_acc=test_step(model=model,
                                     dataloader=test_dataloader,
                                     loss_fn=loss_fn,
                                     accuracy_fn=accuracy_fn,
                                     device=device)
        print(f"Epochs :{epoch} | Train loss :{train_loss:.4} | Train accuracy :{train_acc:.3} | Test loss :{test_loss:.4} | Test accuracy :{test_acc:.3} " )

        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)
    end=timer()
    print(end-start)        
    return results        
        
    

In [38]:
def accuracy_fn(y_true, y_pred):
    return (y_true == y_pred).sum().item() / len(y_true)

In [39]:
optim , loss_fn = create_optim_loss(model = lstm_model,lr = 0.001 ,weight_tensor=weight_tensor,device=DEVICE)

In [40]:
result = Train(model = lstm_model,
               train_dataloader=train_loader,
               test_dataloader=test_loader,
               optimizer=optim ,
               loss_fn=loss_fn,
               accuracy_fn=accuracy_fn,
               epochs = 5 ,
               device =DEVICE)

  0%|          | 0/5 [00:00<?, ?it/s]

  Batch 0, time: 2.3s
  Batch 200, time: 5.5s
  Batch 400, time: 8.5s
  Batch 600, time: 11.6s
Epochs :0 | Train loss :0.5456 | Train accuracy :0.726 | Test loss :0.4665 | Test accuracy :0.795 
  Batch 0, time: 1.8s
  Batch 200, time: 5.1s
  Batch 400, time: 8.2s
  Batch 600, time: 11.4s
Epochs :1 | Train loss :0.4961 | Train accuracy :0.77 | Test loss :0.5073 | Test accuracy :0.777 
  Batch 0, time: 1.8s
  Batch 200, time: 5.0s
  Batch 400, time: 8.1s
  Batch 600, time: 11.3s
Epochs :2 | Train loss :0.4405 | Train accuracy :0.803 | Test loss :0.4502 | Test accuracy :0.804 
  Batch 0, time: 2.1s
  Batch 200, time: 5.2s
  Batch 400, time: 8.3s
  Batch 600, time: 11.4s
Epochs :3 | Train loss :0.4073 | Train accuracy :0.819 | Test loss :0.3981 | Test accuracy :0.827 
  Batch 0, time: 1.8s
  Batch 200, time: 5.1s
  Batch 400, time: 8.2s
  Batch 600, time: 11.3s
Epochs :4 | Train loss :0.3907 | Train accuracy :0.829 | Test loss :0.402 | Test accuracy :0.832 
79.83440989999508


In [41]:
torch.save(lstm_model.state_dict(), '../models/sentiment_model.pth')
print("model saved!")


import pickle
with open('../models/tokenizer_info.pkl', 'wb') as f:
    pickle.dump({'vocab_size': tokenizer.n_vocab}, f)

model saved!
